# EXP-002 — legal-schema prefix validation

## tl;dr

EXP-001 exposed a validation leak: formation columns (`ANCC`…`BUDA`) exist in train horizontal files but not in test horizontal files. This experiment rebuilds the multi-cut benchmark using only features available in both schemas. It deliberately excludes same-well train lookup and leaderboard information.

## Context & Methods

### Key assumptions

- At each outer cut, `TVT` is visible only before the cut.
- Future `MD, X, Y, Z, GR` rows are available, matching test-time inputs.
- Test typewells provide only `TVT, GR`; train-only `Geology` is forbidden.
- Cuts are 50%, 65%, 75%, and 85%.
- Pooled RMSE is primary; median and p90 well-case RMSE are guardrails.

This first legal benchmark evaluates inexpensive extrapolation candidates. Typewell/GR sequence alignment is intentionally deferred to EXP-003 so its incremental contribution can be measured cleanly.

In [1]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

SEED = 42
CUT_FRACTIONS = (0.50, 0.65, 0.75, 0.85)
LOCAL_WINDOW = 300
ALLOWED_HORIZONTAL = ('MD', 'X', 'Y', 'Z', 'GR')
ALLOWED_TYPEWELL = ('TVT', 'GR')
FORBIDDEN = ('ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA', 'Geology')
np.random.seed(SEED)

def find_project_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        q = p / 'competitions/public-comp/wellbore-geology-prediction'
        if (q / 'datasets/train').is_dir(): return q
        if p.name == 'wellbore-geology-prediction' and (p / 'datasets/train').is_dir(): return p
    raise FileNotFoundError('Run from the repository or competition directory')

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / 'datasets'
RESULTS_DIR = PROJECT_ROOT / 'experiments/exp_002_legal_schema_validation/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WELL_FILES = sorted((DATA / 'train').glob('*__horizontal_well.csv'))
assert len(WELL_FILES) == 773
print('project:', PROJECT_ROOT)
print('wells:', len(WELL_FILES))

project: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/wellbore-geology-prediction
wells: 773


## Schema contract

The usable horizontal feature set is defined by the train/test intersection, not by the richer train schema.

In [2]:
train_example = pd.read_csv(WELL_FILES[0], nrows=2)
test_example = pd.read_csv(sorted((DATA / 'test').glob('*__horizontal_well.csv'))[0], nrows=2)
train_tw = pd.read_csv(sorted((DATA / 'train').glob('*__typewell.csv'))[0], nrows=2)
test_tw = pd.read_csv(sorted((DATA / 'test').glob('*__typewell.csv'))[0], nrows=2)

horizontal_intersection = sorted(set(train_example.columns) & set(test_example.columns))
typewell_intersection = sorted(set(train_tw.columns) & set(test_tw.columns))
contract = pd.DataFrame({
    'source': ['horizontal intersection', 'typewell intersection', 'train-only forbidden'],
    'columns': [', '.join(horizontal_intersection), ', '.join(typewell_intersection), ', '.join(FORBIDDEN)],
})
display(contract)
assert set(ALLOWED_HORIZONTAL).issubset(horizontal_intersection)
assert set(ALLOWED_TYPEWELL).issubset(typewell_intersection)
assert not set(FORBIDDEN).intersection(test_example.columns)
assert 'Geology' not in test_tw.columns

,source,columns
0,horizontal intersection,"GR, MD, TVT_input, X, Y, Z"
1,typewell intersection,"GR, TVT"
2,train-only forbidden,"ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA, Geology"


## Legal baseline candidates

- `last_value`: constant continuation.
- `local_md_linear`: local TVT trend versus measured depth.
- `robust_md_slope`: median recent first-difference slope.
- `local_z_linear`: local TVT trend versus vertical coordinate.
- `z_residual_linear`: extrapolates the local trend of `TVT + Z`, then subtracts future `Z`.
- `inner_selector`: chooses among these using a nested holdout inside the visible prefix.

In [3]:
def rmse(y, pred):
    return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(pred, float)) ** 2)))

def candidates(df, n_visible):
    visible, future = df.iloc[:n_visible], df.iloc[n_visible:]
    w = visible.iloc[-min(LOCAL_WINDOW, n_visible):]
    out = {'last_value': np.full(len(future), visible.TVT.iloc[-1])}

    md_coef = np.polyfit(w.MD, w.TVT, 1)
    out['local_md_linear'] = np.polyval(md_coef, future.MD)

    dx, dy = np.diff(w.MD), np.diff(w.TVT)
    valid = np.isfinite(dx) & np.isfinite(dy) & (np.abs(dx) > 1e-9)
    slope = float(np.median(dy[valid] / dx[valid])) if valid.any() else 0.0
    out['robust_md_slope'] = visible.TVT.iloc[-1] + slope * (future.MD.to_numpy() - visible.MD.iloc[-1])

    z_coef = np.polyfit(w.Z, w.TVT, 1)
    out['local_z_linear'] = np.polyval(z_coef, future.Z)

    residual_coef = np.polyfit(w.MD, w.TVT + w.Z, 1)
    out['z_residual_linear'] = np.polyval(residual_coef, future.MD) - future.Z.to_numpy()
    return out

def inner_select(df, n_visible):
    inner = min(max(100, int(n_visible * 0.80)), n_visible - 50)
    preds = candidates(df.iloc[:n_visible].reset_index(drop=True), inner)
    truth = df.TVT.iloc[inner:n_visible]
    scores = {name: rmse(truth, pred) for name, pred in preds.items()}
    return min(scores, key=scores.get)


## Run validation

In [4]:
started = time.time()
records, failures = [], []
for number, path in enumerate(WELL_FILES, 1):
    well = path.name.split('__')[0]
    try:
        raw = pd.read_csv(path).sort_values('MD').reset_index(drop=True)
        # Explicitly discard train-only columns before any candidate sees the frame.
        df = raw[[*ALLOWED_HORIZONTAL, 'TVT']].dropna(subset=['MD', 'Z', 'TVT']).reset_index(drop=True)
        assert not set(FORBIDDEN).intersection(df.columns)
        for cut in CUT_FRACTIONS:
            n_visible = int(round(len(df) * cut))
            truth = df.TVT.iloc[n_visible:].to_numpy(float)
            preds = candidates(df, n_visible)
            selected = inner_select(df, n_visible)
            preds['inner_selector'] = preds[selected]
            for model, pred in preds.items():
                error = np.asarray(pred) - truth
                records.append({
                    'well': well, 'cut_fraction': cut, 'model': model,
                    'n_visible': n_visible, 'n_hidden': len(truth),
                    'rmse': rmse(truth, pred), 'bias': float(error.mean()),
                    'squared_error_sum': float(np.sum(error ** 2)),
                    'selected_model': selected if model == 'inner_selector' else '',
                })
    except Exception as exc:
        failures.append({'well': well, 'reason': repr(exc)})
    if number % 100 == 0: print(f'{number}/{len(WELL_FILES)}')

results = pd.DataFrame(records)
failures_df = pd.DataFrame(failures)
elapsed = time.time() - started
print(f'{results.well.nunique()} wells; {len(results)} metrics; failures={len(failures_df)}; {elapsed:.1f}s')
assert results.well.nunique() == 773 and failures_df.empty
results.head()

100/773


200/773


300/773


400/773


500/773


600/773


700/773


773 wells; 18552 metrics; failures=0; 5.2s


,well,cut_fraction,model,n_visible,n_hidden,rmse,bias,squared_error_sum,selected_model
0,000d7d20,0.5,last_value,2639,2639,4.533036,2.619693,5.422727e+04,
1,000d7d20,0.5,local_md_linear,2639,2639,31.884165,-28.474942,2.682807e+06,
2,000d7d20,0.5,robust_md_slope,2639,2639,26.829309,-23.780307,1.899583e+06,
3,000d7d20,0.5,local_z_linear,2639,2639,22.209365,-20.108914,1.301702e+06,
4,000d7d20,0.5,z_residual_linear,2639,2639,13.568012,-12.437226,4.858160e+05,


## Results

In [5]:
def summarize(g):
    return pd.Series({
        'pooled_rmse': np.sqrt(g.squared_error_sum.sum() / g.n_hidden.sum()),
        'mean_well_rmse': g.rmse.mean(),
        'median_well_rmse': g.rmse.median(),
        'p90_well_rmse': g.rmse.quantile(.90),
        'worst_well_rmse': g.rmse.max(),
        'well_cases': len(g),
        'hidden_rows': g.n_hidden.sum(),
    })

summary = results.groupby('model', sort=False).apply(summarize, include_groups=False).reset_index().sort_values('pooled_rmse')
by_cut = results.groupby(['cut_fraction', 'model'], sort=False).apply(summarize, include_groups=False).reset_index()
winners = results.loc[results.groupby(['well', 'cut_fraction']).rmse.idxmin()].groupby('model').size().rename('wins').reset_index().sort_values('wins', ascending=False)
selector_choices = results[results.model == 'inner_selector'].selected_model.value_counts().rename_axis('selected_model').reset_index(name='cases')
display(summary.round(4))
display(by_cut.pivot(index='model', columns='cut_fraction', values='pooled_rmse').round(4))
display(winners)
display(selector_choices)

,model,pooled_rmse,mean_well_rmse,median_well_rmse,p90_well_rmse,worst_well_rmse,well_cases,hidden_rows
0,last_value,11.4937,8.6360,7.0537,16.8669,58.1652,3092.0,6365301.0
4,z_residual_linear,21.2214,12.3499,8.2177,27.8373,161.6781,3092.0,6365301.0
5,inner_selector,21.3044,11.8730,7.8759,25.3180,313.3661,3092.0,6365301.0
2,robust_md_slope,31.2987,18.4268,12.1532,41.0586,240.9127,3092.0,6365301.0
1,local_md_linear,31.6612,18.7576,12.4237,42.1399,263.1030,3092.0,6365301.0
3,local_z_linear,66.7216,25.0682,12.0212,50.5762,1073.0782,3092.0,6365301.0


cut_fraction,0.50,0.65,0.75,0.85
model,,,,
inner_selector,24.5316,22.5254,17.4276,9.2849
last_value,12.6456,11.5766,10.4752,8.4999
local_md_linear,40.9969,28.1763,20.5643,13.9237
local_z_linear,88.9923,53.2789,45.7477,24.2450
robust_md_slope,40.8320,27.2802,20.2491,13.6404
z_residual_linear,27.1880,19.4858,14.3086,7.8594


,model,wins
0,last_value,1356
4,z_residual_linear,1114
1,local_md_linear,247
2,local_z_linear,241
3,robust_md_slope,134


,selected_model,cases
0,z_residual_linear,1470
1,last_value,1027
2,local_md_linear,220
3,local_z_linear,214
4,robust_md_slope,161


## Takeaways

This is the first deployable-schema validation anchor. EXP-003 should add typewell/GR sequence alignment and the current PF/beam candidates one at a time, preserving this exact split and reporting both pooled RMSE and p90 well-case RMSE.

In [6]:
results.to_csv(RESULTS_DIR / 'per_well_cut_metrics.csv', index=False)
summary.to_csv(RESULTS_DIR / 'summary.csv', index=False)
by_cut.to_csv(RESULTS_DIR / 'summary_by_cut.csv', index=False)
winners.to_csv(RESULTS_DIR / 'winner_counts.csv', index=False)
selector_choices.to_csv(RESULTS_DIR / 'selector_choices.csv', index=False)
failures_df.to_csv(RESULTS_DIR / 'failures.csv', index=False)
contract.to_csv(RESULTS_DIR / 'schema_contract.csv', index=False)
run = {
    'experiment_id': 'exp_002', 'seed': SEED, 'cuts': list(CUT_FRACTIONS),
    'wells': int(results.well.nunique()), 'metrics': int(len(results)),
    'elapsed_sec': elapsed, 'best_model': str(summary.iloc[0].model),
    'best_pooled_rmse': float(summary.iloc[0].pooled_rmse),
}
(RESULTS_DIR / 'run.json').write_text(json.dumps(run, indent=2) + '\n')
print(json.dumps(run, indent=2))
print('saved:', sorted(p.name for p in RESULTS_DIR.iterdir()))

{
  "experiment_id": "exp_002",
  "seed": 42,
  "cuts": [
    0.5,
    0.65,
    0.75,
    0.85
  ],
  "wells": 773,
  "metrics": 18552,
  "elapsed_sec": 5.2484307289123535,
  "best_model": "last_value",
  "best_pooled_rmse": 11.493684387681856
}
saved: ['failures.csv', 'per_well_cut_metrics.csv', 'run.json', 'schema_contract.csv', 'selector_choices.csv', 'summary.csv', 'summary_by_cut.csv', 'winner_counts.csv']
